# Rossmann Store Sales 


<img src='https://cdn.rossmann.com.tr/media/wysiwyg/rossmann_magaza.jpeg'>


Bu çalışmada `Rossmann Store Sales` yarışması için mağaza ve tarih bilgileri kullanılarak günlük satış tahmini yapan bir time series başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Veri birleştirme ve ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde satış tahmini ve veri analizi için gerekli kütüphaneleri içe aktarıyoruz.


In [2]:
from google.colab import drive
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [ ]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/rossmann-store-sales.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, 'train.csv')
test_member = find_zip_member(zip_members, 'test.csv')
store_member = find_zip_member(zip_members, 'store.csv')
sample_submission_member = find_zip_member(zip_members, 'sample_submission.csv')


## 3. Veri Dosyalarını Yükleme


In [6]:
def read_csv_from_zip(zip_path, member_name, **kwargs):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            return pd.read_csv(f, **kwargs)

train = read_csv_from_zip(zip_path, train_member)
test = read_csv_from_zip(zip_path, test_member)
store = read_csv_from_zip(zip_path, store_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Store shape:', store.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


/tmp/ipykernel_28690/1261770574.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(f, **kwargs)


Train shape: (1017209, 9)
Test shape: (41088, 8)
Store shape: (1115, 10)
Sample submission shape: (41088, 2)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


## 4. Veri Birleştirme ve Ön İşleme


In [7]:
# Bu bölümde train ve test tablolarını mağaza bilgileri ile birleştiriyoruz.


In [8]:
train_df = train.merge(store, on='Store', how='left')
test_df = test.merge(store, on='Store', how='left')

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

print('Train merged shape:', train_df.shape)
print('Test merged shape:', test_df.shape)
train_df.head()


Train merged shape: (1017209, 18)
Test merged shape: (41088, 17)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [9]:
# Bu yarışmada kapalı günlerde satış sıfır olabildiği için modelleme öncesinde bu davranışın etkisini göreceğiz.


## 5. Özellik Çıkarımı


In [10]:
# Bu bölümde tarih bilgisinden ek alanlar çıkarıyoruz ve model girdilerini hazırlıyoruz.


In [11]:
for df in [train_df, test_df]:
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['DayOfWeekFromDate'] = df['Date'].dt.dayofweek
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
    df['IsWeekend'] = df['DayOfWeekFromDate'].isin([5, 6]).astype(int)

feature_columns = [
    'Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday',
    'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear',
    'PromoInterval', 'Year', 'Month', 'Day', 'DayOfWeekFromDate', 'WeekOfYear', 'IsWeekend'
]

x = train_df[feature_columns].copy()
y = train_df['Sales'].copy()

test_x = test_df[feature_columns].copy()

categorical_features = ['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']

for col in categorical_features:
    x[col] = x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42
)

numerical_features = [col for col in x.columns if col not in categorical_features]

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (813767, 21)
x_valid shape: (203442, 21)


## 6. Model Kurma


In [12]:
# Bu bölümde sayısal ve kategorik alanları hazırlayıp satış tahmini yapan başlangıç modelini kuruyoruz.


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=18,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(x_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Store', 'DayOfWeek', 'Open',
                                                   'Promo', 'SchoolHoliday',
                                                   'CompetitionDistance',
                                                   'CompetitionOpenSinceMonth',
                                                   'CompetitionOpenSinceYear',
                                                   'Promo2', 'Promo2SinceWeek',
                                                   'Promo2SinceYear', 'Year',
                                                   'Month', 'Day',
                                                   'DayOfWeekFromDate',
                                                   'WeekOfYear', 'IsWeekend']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['StateHoliday', 'StoreType',
                                                   'Assortment',
                                                   'PromoInterval'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=18, min_samples_split=5,
                                       n_estimators=200, n_jobs=-1,
                                       random_state=42))])

## 7. RMSE Değerlendirmesi


In [14]:
# Bu bölümde modelin doğrulama setindeki hata düzeyini RMSE ile ölçüyoruz.


In [15]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(y_valid, valid_preds)
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 1326.99495


## 8. Test Tahmini ve Submission


In [16]:
# Bu bölümde test seti için satış tahminlerini üretip submission dosyasını oluşturuyoruz.


In [17]:
submission = sample_submission.copy()
submission['Sales'] = model.predict(test_x)

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (41088, 2)


,Id,Sales
0,1,5681.278549
1,2,7922.375095
2,3,8009.740294
3,4,6702.727106
4,5,7526.337507


In [18]:
output_path = '/content/rossmann_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/rossmann_submission.csv


## 9. Sonuç


Bu çalışmada Rossmann Store Sales yarışması için mağaza ve tarih bilgileri birleştirilerek günlük satış tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 1326.99495 RMSE değeri elde etti ve test verisi için günlük satış tahminleri üretildi.
